SummarizationMidlleware中间件 对历史消息列表进行摘要和总结，达到压缩上下文的效果
达到触发条件时，将结果作为HumanMessage放到消息列表最开始

model 用于摘要的模型 2trigger 触发条件 3个条件都可以写 token、message、fraction（上下文长度的比例，历史累计token达到模型的Max_input_tokens*fraction的触发摘要）任意一个触发 3 keep 摘要时保留的原始消息 token、message、fraction 同一时间只接受1种条件 4 token_counter token量统计函数 5 summary_prompt摘要时自定义提示词 7 trim_token_to_summarize 摘要时历史消息最大Token数量，默认为4000 历史消息token数大于该值的时候就会被裁剪

In [22]:

from dataclasses import dataclass
from idlelib.undo import Command
from typing import Literal, TypedDict, Annotated, Union
from urllib import response

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware, \
    ContextEditingMiddleware, ClearToolUsesEdit
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy, \
    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os

from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool

load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
myAgent=create_agent(
    model=myModel,
    name="agent01",
    system_prompt="你是1个专业的旅游小助手,当有人问你关于旅游计划的时候，你会用各种天气查询工具、旅游景点参考网址、美团上当地的特色美食评价、住宿信息评价、和当地的交通中心网站查询各种信息",
    middleware=[
        SummarizationMiddleware(
            model=myModel,
            trigger=[  ("tokens",100),  ("messages",4),  ],
            keep=("messages",4),
            summary_prompt=(
                "你是一个对话摘要助手。请将以下历史对话压缩为一段简洁的中文摘要。"
                "要求："
                "1. 仅输出摘要正文，不要包含Here is a summary of the conversation to date这句话"
                "2. 保留关键实体（人名、地名、时间、意图）。"
                "3. 字数控制在100字以内。"
                "历史对话：\n{messages}"
             )
        )
    ]
   )
#5 调用agent
for chunk in myAgent.stream(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                SystemMessage("你是1个专业的旅游假小助手,当有人问你关于天气问题的时候，如果你自己不知道，可以查询中国的气象中心发布的最新消息。")  ,
                HumanMessage("你好哇，我是Aureliano"),
                AIMessage("你好，很高兴认识你"),
                HumanMessage("我想知道明天的天气怎么样"),
                AIMessage("请告诉我具体的城市和时间"),
                HumanMessage("我在重庆江北区"),
                AIMessage("时间呢"),
                HumanMessage("今天是2026-9-18，我想查询下2026-9-19的"),
                AIMessage("好的，这就为您查询重庆市江北区2026-9-19的天气预报信息"),
            ]
    },
    stream_mode="values"
):
    rprint(chunk)
    print("--"*50)

{
    'messages': [
        SystemMessage(
            content='你是1个专业的旅游假小助手,当有人问你关于天气问题的时候，如果你自己不知道，可以查询中国的气象中
心发布的最新消息。',
            additional_kwargs={},
            response_metadata={},
            id='788042c3-b9e0-4cab-a000-03e7f02f8f36'
        ),
        HumanMessage(
            content='你好哇，我是Aureliano',
            additional_kwargs={},
            response_metadata={},
            id='9ba86808-0b43-4f62-9a45-b62d8b13f4aa'
        ),
        AIMessage(
            content='你好，很高兴认识你',
            additional_kwargs={},
            response_metadata={},
            id='d6cf7379-d3ec-476e-9f88-e41fce3ccf94',
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(
            content='我想知道明天的天气怎么样',
            additional_kwargs={},
            response_metadata={},
            id='4fbcf1fc-e354-4f0b-9444-4310fb67a861'
        ),
        AIMessage(
            content='请告诉我具体的城市和时间',
            additional_kwargs={},
            response_metadata={},
            id='9add71af-52e5-4927-aa7d-3f930011a92d',
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(
            content='我在重庆江北区',
            additional_kwargs={},
            response_metadata={},
            id='0899ad7d-07a5-4478-a7d6-a2998c03daca'
        ),
        AIMessage(
            content='时间呢',
            additional_kwargs={},
            response_metadata={},
            id='efadcffc-98de-4d2c-9980-2260aedfb9a6',
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(
            content='今天是2026-9-18，我想查询下2026-9-19的',
            additional_kwargs={},
            response_metadata={},
            id='8d945298-5ac4-4a7c-8cbb-7e6e5be696c9'
        ),
        AIMessage(
            content='好的，这就为您查询重庆市江北区2026-9-19的天气预报信息',
            additional_kwargs={},
            response_metadata={},
            id='a845d704-8ed1-458b-a905-13d0e9f2a03a',
            tool_calls=[],
            invalid_tool_calls=[]
        )
    ]
}

----------------------------------------------------------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='Here is a summary of the conversation to 
date:\n\nAureliano问候并询问明天天气，助手请其提供具体城市和时间以便查询。',
            additional_kwargs={'lc_source': 'summarization'},
            response_metadata={},
            id='8a20d5bb-cff2-4dd4-a228-dabfd33b7bb0'
        ),
        HumanMessage(
            content='我在重庆江北区',
            additional_kwargs={},
            response_metadata={},
            id='0899ad7d-07a5-4478-a7d6-a2998c03daca'
        ),
        AIMessage(
            content='时间呢',
            additional_kwargs={},
            response_metadata={},
            id='efadcffc-98de-4d2c-9980-2260aedfb9a6',
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(
            content='今天是2026-9-18，我想查询下2026-9-19的',
            additional_kwargs={},
            response_metadata={},
            id='8d945298-5ac4-4a7c-8cbb-7e6e5be696c9'
        ),
        AIMessage(
            content='好的，这就为您查询重庆市江北区2026-9-19的天气预报信息',
            additional_kwargs={},
            response_metadata={},
            id='a845d704-8ed1-458b-a905-13d0e9f2a03a',
            tool_calls=[],
            invalid_tool_calls=[]
        )
    ]
}

----------------------------------------------------------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='Here is a summary of the conversation to 
date:\n\nAureliano问候并询问明天天气，助手请其提供具体城市和时间以便查询。',
            additional_kwargs={'lc_source': 'summarization'},
            response_metadata={},
            id='8a20d5bb-cff2-4dd4-a228-dabfd33b7bb0'
        ),
        HumanMessage(
            content='我在重庆江北区',
            additional_kwargs={},
            response_metadata={},
            id='0899ad7d-07a5-4478-a7d6-a2998c03daca'
        ),
        AIMessage(
            content='时间呢',
            additional_kwargs={},
            response_metadata={},
            id='efadcffc-98de-4d2c-9980-2260aedfb9a6',
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(
            content='今天是2026-9-18，我想查询下2026-9-19的',
            additional_kwargs={},
            response_metadata={},
            id='8d945298-5ac4-4a7c-8cbb-7e6e5be696c9'
        ),
        AIMessage(
            content='好的，这就为您查询重庆市江北区2026-9-19的天气预报信息',
            additional_kwargs={},
            response_metadata={},
            id='a845d704-8ed1-458b-a905-13d0e9f2a03a',
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        AIMessage(
            content='<tool_call>\n{"name": "weather_forecast", "arguments": {"city": "重庆江北区", "date": 
"2026-09-19"}}\n</tool_call>\n<tool_call>\n{"name": "weather_forecast", "arguments": {"city": "重庆市江北区", 
"date": "2026-09-19"}}\n</tool_call>',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 193,
                    'prompt_tokens': 203,
                    'total_tokens': 396,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 113,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 193
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 203
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-ba095274-d34b-94c0-a1db-965efc962f9c',
                'finish_reason': 'stop',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0b299-05a8-72a3-801a-3bb0ad276b1a-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 203,
                'output_tokens': 193,
                'total_tokens': 396,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 113}
            }
        )
    ]
}

----------------------------------------------------------------------------------------------------


HumanInTheLoopMiddleware中间件 人在环中间件，在工具调用之前中断Agent运行，等待用户对工具的调用请求决策：1 approve 2 edit编辑调用配置后再执行 3 reject

参数说明 1 interrupt_on  设置在指定工具运行之前进行中断，以及中断的策略是什么，可以选择的策略有 Ture False，或者InterruptOnConfig对象，类似于自定义策略 2 description_prefix 自定义中断的工具的描述

In [41]:

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware,HumanInTheLoopMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
class WeatherInput(BaseModel):
    #如果在函数声明处指定了参数以及参数类型，并且启用了args_schema，那么则以args_schema指定的类型为准
    city: str=Field(
        description="具体的城市名称，例如'北京市'、'重庆市'、''天津市",
        default="北京市",
        min_length=2,
        max_length=10,
    )
    unit:Literal["celsius","fahrenheit","kelvin"]=Field(
        description="描述气温的单位，必须是摄氏度，凯文等单位"
    )
    if_forcast:bool=Field(
        default=False,
        description="是否预测未来1天的天气"
    )
#3 声明1个函数（工具）
@tool(description="获取具体城市的天气",args_schema=WeatherInput)
def get_weather(city:str,unit:str,if_forcast:bool):
    result= f"2026-09-18{city}天气晴朗,9-16{unit},温度适宜，适合出门晒太阳"
    if if_forcast:
        result+="但是明天可能有雨，记得带伞"
    return result
#4 将函数绑定在模型上
@tool(description="查询当天的新闻")
def get_news():
    """
    查询当天的新闻
    """
    return "赛力斯华为分手，赛力斯股价堪忧"
@tool(description="用于发送邮件给自己的邮箱")
def send_to_my_email():
    """
    用于发送邮件给自己的邮箱
    """
    return "已成功发送邮件"
myConfig={"configurable":{"thread_id":"1"}}
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[get_weather,get_news,send_to_my_email],
    system_prompt="你是1个专业的个人助手，查询消息的时候会使用已知的工具",
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_news":False,
                "get_weather":True,
                "send_to_my_email":{
                    "allowed_decisions":["approve","reject"],
                    "description":"邮件发送过程中断！！！"
                }
            },
            description_prefix="中断啦！！！！！！！！！！"
        )
    ]
   )
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你好哇，我想查询下2026-09-18的天气和新闻，然后发一份邮件给自己"),
            ],

    },
       config=myConfig,
)
rprint(response)
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware,HumanInTheLoopMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
class WeatherInput(BaseModel):
    #如果在函数声明处指定了参数以及参数类型，并且启用了args_schema，那么则以args_schema指定的类型为准
    city: str=Field(
        description="具体的城市名称，例如'北京市'、'重庆市'、''天津市",
        default="北京市",
        min_length=2,
        max_length=10,
    )
    unit:Literal["celsius","fahrenheit","kelvin"]=Field(
        description="描述气温的单位，必须是摄氏度，凯文等单位"
    )
    if_forcast:bool=Field(
        default=False,
        description="是否预测未来1天的天气"
    )
#3 声明1个函数（工具）
@tool(description="获取具体城市的天气",args_schema=WeatherInput)
def get_weather(city:str,unit:str,if_forcast:bool):
    result= f"2026-09-18{city}天气晴朗,9-16{unit},温度适宜，适合出门晒太阳"
    if if_forcast:
        result+="但是明天可能有雨，记得带伞"
    return result
#4 将函数绑定在模型上
@tool(description="查询当天的新闻")
def get_news():
    """
    查询当天的新闻
    """
    return "赛力斯华为分手，赛力斯股价堪忧"
@tool(description="用于发送邮件给自己的邮箱")
def send_to_my_email():
    """
    用于发送邮件给自己的邮箱
    """
    return "已成功发送邮件"
myConfig={"configurable":{"thread_id":"1"}}
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[get_weather,get_news,send_to_my_email],
    system_prompt="你是1个专业的个人助手，查询消息的时候会使用已知的工具",
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_news":False,
                "get_weather":True,
                "send_to_my_email":{
                    "allowed_decisions":["approve","reject"],
                    "description":"邮件发送过程中断！！！"
                }
            },
            description_prefix="中断啦！！！！！！！！！！"
        )
    ]
   )
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你好哇，我想查询下2026-09-18的天气和新闻，然后发一份邮件给自己"),
            ],

    },
       config=myConfig,
)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='你好哇，我想查询下2026-09-18的天气和新闻，然后发一份邮件给自己',
            additional_kwargs={},
            response_metadata={},
            id='075feffb-97e1-4cc6-a5ec-44215dfdcc11'
        ),
        AIMessage(
            content='你好哇！😊 
我来帮你查询天气和新闻。\n\n不过先说明一下：天气工具只支持查询**当前天气**或**未来1天的预报**，没法直接指定 
2026-09-18 
这个具体日期。我会先按默认的北京市帮你查询，同时打开未来预报功能，你看看是否符合预期～\n\n让我同时查询天气和新闻：\
n\n',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 392,
                    'prompt_tokens': 531,
                    'total_tokens': 923,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 242,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 392
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 531
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-09572425-3245-9891-978c-623c38a04d4e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0b3b5-3f19-7e63-b602-e47605f60ffd-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京市', 'unit': 'celsius', 'if_forcast': True},
                    'id': 'call_d44c7b7def3442ee8d90d165',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_c3781c979a0c4d188030075d', 'type': 'tool_call'}
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 531,
                'output_tokens': 392,
                'total_tokens': 923,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 242}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京市', 'unit': 'celsius', 'if_forcast': True},
                        'description': "中断啦！！！！！！！！！！\n\nTool: get_weather\nArgs: {'city': '北京市', 
'unit': 'celsius', 'if_forcast': True}"
                    }
                ],
                'review_configs': [
                    {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}
                ]
            },
            id='293c6336a66580a818f8e52aac999f19'
        )
    ]
}

In [42]:
from langgraph.types import Command
weather_decision={
    "type":"edit",
    "edited_action":{
        "name":"get_weather",
        "args":{
            "city":"重庆市",
            "if_forcast":False,
        }

    }
}
email_decision={
    "type":"approve"
}
decisions={
    "decisions":[]

}

interrupts=response.get("__interrupt__",[])
action_requests=interrupts[0].value["action_requests"]
for action_request in action_requests:
    if action_request["name"]=="get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"]=="send_to_my_email":
        decisions["decisions"].append(email_decision)

if interrupts:
   resumed_response= myAgent.invoke(
        Command(resume=decisions),
        config=myConfig,
    )
   for msg in resumed_response["messages"]:
       msg.pretty_print()

================================ Human Message =================================

你好哇，我想查询下2026-09-18的天气和新闻，然后发一份邮件给自己
================================== Ai Message ==================================
Name: agent01

你好哇！😊 我来帮你查询天气和新闻。

不过先说明一下：天气工具只支持查询**当前天气**或**未来1天的预报**，没法直接指定 2026-09-18 这个具体日期。我会先按默认的北京市帮你查询，同时打开未来预报功能，你看看是否符合预期～

让我同时查询天气和新闻：
Tool Calls:
  get_weather (call_d44c7b7def3442ee8d90d165)
 Call ID: call_d44c7b7def3442ee8d90d165
  Args:
    city: 重庆市
    if_forcast: False
  get_news (call_c3781c979a0c4d188030075d)
 Call ID: call_c3781c979a0c4d188030075d
  Args:
================================= Tool Message =================================
Name: get_weather

Error invoking tool 'get_weather' with kwargs {'city': '重庆市', 'if_forcast': False} with error:
 unit: Field required
 Please fix the error and try again.
================================= Tool Message =================================
Name: get_news

赛力斯华为分手，赛力斯股价堪忧
================================== Ai Message =====

PIIMiddleware中间件 用于检测和处理对话中的个人信息

参数1 pii_type 检测的数据类型(email,credit_card,url,mac_address,ip) 参数2 strategy 处理下Pii信息的策略 redact策略完全擦除和隐藏，适合日志清洗，合规需求，公开输出时隐藏敏感内容 ；mask策略 部分隐藏，又保留一点可辨识性，适合用户服务界面，前端显示的场景 ；hash策略，用哈希值替代原值，适合分析、调试、统计分析、匿名追踪等场景；block策略，检测到PII信息直接抛异常。适合隐私要求极高、绝对不允许泄露任何敏感信息的场景

参数3 detector 自定义检测PII值的函数，或者正则表达式 参数4 apply_to_input默认为True，是否在调用模型前检测 ；参数5 apply_to_output 默认为false，是否在调用模型后检测；参数6 apply_to_tool_results 默认为False 是否在工具调用后检测其输出内容

In [45]:

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware,HumanInTheLoopMiddleware,PIIMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)

# myConfig={"configurable":{"thread_id":"1"}}
myAgent=create_agent(
    model=myModel,
    name="agent01",
    # system_prompt="你是1个专业的个人助手，查询消息的时候会使用已知的工具",
    # checkpointer=InMemorySaver(),
    middleware=[
        PIIMiddleware("email",strategy="redact",apply_to_input=True),
        PIIMiddleware("credit_card",strategy="mask",apply_to_input=True),
        PIIMiddleware("url",strategy="hash",apply_to_input=True),
        PIIMiddleware("ip",strategy="hash",apply_to_input=True),
        PIIMiddleware("mac_address",strategy="mask",apply_to_input=True)
    ]
   )

response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你好哇，发一份邮件给自己，邮件地址是wei.xiang@serses.com,内容是："
                             "credit_card:0088155467"
                             "我的个人网址: https://www.3cschool.com"
                             "ip是10.0.15.66"
                             "主机地址是 E5:56:A3:DD:F3:66"),
            ],

    },
       # config=myConfig,
)
for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

你好哇，发一份邮件给自己，邮件地址是wei[REDACTED_EMAIL],内容是：credit_card:0088155467我的个人网址: <url_hash:55dca27b> **:**:**:**:**:66
================================== Ai Message ==================================
Name: agent01

你好！我无法发送邮件，因为我没有发送邮件的功能。我是一个对话式 AI，只能进行文字对话。

另外，温馨提示：你分享的内容中包含信用卡号等敏感信息，建议不要在对话中透露此类信息，注意保护个人隐私和财产安全。

如果你需要发送邮件，可以使用你自己的邮箱客户端（如 Outlook、Gmail 等）来操作。有其他我能帮到的事情吗？


In [48]:
myAgent=create_agent(
    model=myModel,
    name="agent01",
    # system_prompt="你是1个专业的个人助手，查询消息的时候会使用已知的工具",
    # checkpointer=InMemorySaver(),
    middleware=[
        PIIMiddleware("email",strategy="redact",apply_to_input=True),
        PIIMiddleware("credit_card",strategy="mask",apply_to_input=True),
        PIIMiddleware("url",strategy="hash",apply_to_input=True),
        PIIMiddleware("ip",strategy="hash",apply_to_input=True),
        PIIMiddleware("mac_address",strategy="block",apply_to_input=True)
    ]
   )
try:
    response=myAgent.invoke(
        {
            # 严格校验，使用message少了1个s就会报错
            "messages":
                [
                    HumanMessage("你好哇，发一份邮件给自己，邮件地址是wei.xiang@serses.com,内容是："
                                 "credit_card:0088155467"
                                 "我的个人网址: https://www.3cschool.com"
                                 "ip是10.0.15.66"
                                 "主机地址是 E5:56:A3:DD:F3:66"),
                ],

        },
           # config=myConfig,
    )
    for msg in response["messages"]:
        msg.pretty_print()
except Exception as e:
    print(f"检测到mac_address,抛出异常信息: {e}")

检测到mac_address,抛出异常信息: Detected 1 instance(s) of mac_address in text content


ToDoListMiddleware中间件 赋予了agent任务规划和追踪进度的能力，可以应对复杂的多步任务  适合的场景：1 任务链路长、步骤多、且有严格先后依赖关系 2 需要在UI界面实时展示agent的思考与执行进度

1system_prompt  自定义指导todo列表使用的提示词，不提供则使用内置提示词，通常不必提供 2 tool_description自定义指导write_tool工具的描述信息，不提供则使用内置描述，通常不必提供

ModelCallLimitMiddleware 中间件 限制模型调用次数，避免无限循环，控制调用成本

1 整个会话的限制，优雅退出

In [50]:

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,     ModelCallLimitMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)

myConfig={"configurable":{"thread_id":"1"}}
myAgent=create_agent(
    model=myModel,
    name="agent01",
    # system_prompt="你是1个专业的个人助手，查询消息的时候会使用已知的工具",
    checkpointer=InMemorySaver(),
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2,#每个线程最多2次模型调用
            # run_limit=5,#每次运行最多5次
            exit_behavior="end",
        )#达到限制后就退出
    ]
   )
print("--"*50,"第一次调用开始","--"*50)
response1=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你好哇")
            ],

    },
       config=myConfig,
)
for msg in response1["messages"]:
    msg.pretty_print()
print("--"*50,"第一次调用结束","--"*50)

print("--"*50,"第二次调用开始","--"*50)
response2=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("很高兴认识你")
            ],

    },
       config=myConfig,
)
for msg in response2["messages"]:
    msg.pretty_print()
print("--"*50,"第二次调用结束","--"*50)

print("--"*50,"第三次调用开始","--"*50)
response3=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你能介绍一下自己吗")
            ],

    },
       config=myConfig,
)
for msg in response3["messages"]:
    msg.pretty_print()
print("--"*50,"第三次调用结束","--"*50)



---------------------------------------------------------------------------------------------------- 第一次调用开始 ----------------------------------------------------------------------------------------------------
================================ Human Message =================================

你好哇
================================== Ai Message ==================================
Name: agent01

你好呀！很高兴见到你～有什么我可以帮你的吗？
---------------------------------------------------------------------------------------------------- 第一次调用结束 ----------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------- 第二次调用开始 ----------------------------------------------------------------------------------------------------
================================ Human Message =================================

你好哇
================================== Ai Message =================================

举例2 整个会话的限制 抛出异常

In [51]:

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,     ModelCallLimitMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)

myConfig={"configurable":{"thread_id":"1"}}
myAgent=create_agent(
    model=myModel,
    name="agent01",
    # system_prompt="你是1个专业的个人助手，查询消息的时候会使用已知的工具",
    checkpointer=InMemorySaver(),
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2,#每个线程最多2次模型调用
            # run_limit=5,#每次运行最多5次
            exit_behavior="error",
        )#达到限制后就退出
    ]
   )
print("--"*50,"第一次调用开始","--"*50)
response1=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你好哇")
            ],

    },
       config=myConfig,
)
for msg in response1["messages"]:
    msg.pretty_print()
print("--"*50,"第一次调用结束","--"*50)

print("--"*50,"第二次调用开始","--"*50)
response2=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("很高兴认识你")
            ],

    },
       config=myConfig,
)
for msg in response2["messages"]:
    msg.pretty_print()
print("--"*50,"第二次调用结束","--"*50)

print("--"*50,"第三次调用开始","--"*50)
response3=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你能介绍一下自己吗")
            ],

    },
       config=myConfig,
)
for msg in response3["messages"]:
    msg.pretty_print()
print("--"*50,"第三次调用结束","--"*50)



---------------------------------------------------------------------------------------------------- 第一次调用开始 ----------------------------------------------------------------------------------------------------
================================ Human Message =================================

你好哇
================================== Ai Message ==================================
Name: agent01

你好哇！很高兴见到你～有什么想聊的或需要帮忙的吗？
---------------------------------------------------------------------------------------------------- 第一次调用结束 ----------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------- 第二次调用开始 ----------------------------------------------------------------------------------------------------
================================ Human Message =================================

你好哇
================================== Ai Message ==============================

ModelCallLimitExceededError: Model call limits exceeded: thread limit (2/2)

举例3 单词调用限制 优雅退出

In [ ]:

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,     ModelCallLimitMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)

myConfig={"configurable":{"thread_id":"1"}}
myAgent=create_agent(
    model=myModel,
    name="agent01",
    # system_prompt="你是1个专业的个人助手，查询消息的时候会使用已知的工具",
    checkpointer=InMemorySaver(),
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2,#每个线程最多2次模型调用
            # run_limit=5,#每次运行最多5次
            exit_behavior="error",
        )#达到限制后就退出
    ]
   )
print("--"*50,"第一次调用开始","--"*50)
response1=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你好哇")
            ],

    },
       config=myConfig,
)
for msg in response1["messages"]:
    msg.pretty_print()
print("--"*50,"第一次调用结束","--"*50)

print("--"*50,"第二次调用开始","--"*50)
response2=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("很高兴认识你")
            ],

    },
       config=myConfig,
)
for msg in response2["messages"]:
    msg.pretty_print()
print("--"*50,"第二次调用结束","--"*50)

print("--"*50,"第三次调用开始","--"*50)
response3=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你能介绍一下自己吗")
            ],

    },
       config=myConfig,
)
for msg in response3["messages"]:
    msg.pretty_print()
print("--"*50,"第三次调用结束","--"*50)



ToolCallLimitMiddleware 中间件，限制工具调用次数的中间件，可以限制总次数，也可以单独限制某个工具的调用次数

退出行为通常有3种模式 1 error直接抛出异常 2 end 结束整个会话 3 continue 继续运行，这是默认行为。此时Agent会将工具调用次数超限的信息发给模型，让模型自助决定是后续行为

ModelFallbackMiddleware中间件 ，用于故障转移，当主模型无法访问的时候，启用备用模型

In [59]:

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-fake",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
secend_model = init_chat_model(
    model="qwen3.8-flash",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)

myConfig={"configurable":{"thread_id":"1"}}
myAgent=create_agent(
    model=myModel,
    name="agent01",
    # system_prompt="你是1个专业的个人助手，查询消息的时候会使用已知的工具",
    checkpointer=InMemorySaver(),
    middleware=[
        ModelFallbackMiddleware(
            secend_model
        )
    ]
   )
print("--"*50,"第一次调用开始","--"*50)
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你好哇,你是千问下的哪个模型，请告诉我你具体的型号，例如qwen3.8-max")
            ],

    },
       config=myConfig,
)
last_msg=response["messages"][-1]
rprint(last_msg)
# for msg in response["messages"]:
#     msg.pretty_print()
print(last_msg.response_metadata.get("model_name"))
print("--"*50,"第一次调用结束","--"*50)





---------------------------------------------------------------------------------------------------- 第一次调用开始 ----------------------------------------------------------------------------------------------------


AIMessage(
    content='你好呀！关于你的问题，我得先跟你说明一下：**“qwen3.8-max” 
并不是一个真实存在的型号哦。**\n\n通义千问（Qwen）是由阿里巴巴集团通义实验室独立开发的大语言模型。目前我能够告诉你
的是，我是 **Qwen** 
的最新版本，我的核心能力涵盖了更高质量的知识语料、深度优化的逻辑推理以及原生多模态处理技术等，但具体的内部型号或版
本号标识，我暂时无法透露给你。\n\n你可以把我当成你贴心的 AI 
助手，有什么需要解答的问题或者想聊的话题，随时都可以找我！',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 373,
            'prompt_tokens': 84,
            'total_tokens': 457,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 246,
                'rejected_prediction_tokens': None,
                'text_tokens': 373
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 0,
                'image_tokens': None,
                'text_tokens': 84
            }
        },
        'model_provider': 'openai',
        'model_name': 'qwen3.8-flash',
        'system_fingerprint': None,
        'id': 'chatcmpl-7f5bdef7-af55-933b-bd09-c712d12bf0e5',
        'finish_reason': 'stop',
        'logprobs': None
    },
    name='agent01',
    id='lc_run--01a0b3f9-e7f8-7ba1-b3f4-abaa0e25b28b-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 84,
        'output_tokens': 373,
        'total_tokens': 457,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 246}
    }
)

qwen3.8-flash
---------------------------------------------------------------------------------------------------- 第一次调用结束 ----------------------------------------------------------------------------------------------------


LLMToolSelectorMiddleware中间件 当工具太多时，用子模型筛选最相关的几个工具

参数1 model用于工具筛选的子模型 2 max_tools 限制可以调用的工具总数量 3 always_include指定的工具不参与max_tools的统计

In [62]:

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
tool_model = init_chat_model(
    model="qwen3.8-flash",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
#实例化的1个对象 工具选择类中间件
tool_selector=LLMToolSelectorMiddleware(
    model=tool_model,
    max_tools=1,
    always_include=["get_weather"]
)

myConfig={"configurable":{"thread_id":"1"}}

class WeatherInput(BaseModel):
    #如果在函数声明处指定了参数以及参数类型，并且启用了args_schema，那么则以args_schema指定的类型为准
    city: str=Field(
        description="具体的城市名称，例如'北京市'、'重庆市'、''天津市",
        default="北京市",
        min_length=2,
        max_length=10,
    )
    unit:Literal["celsius","fahrenheit","kelvin"]=Field(
        description="描述气温的单位，必须是摄氏度，凯文等单位"
    )
    if_forcast:bool=Field(
        default=False,
        description="是否预测未来1天的天气"
    )
#3 声明1个函数（工具）
@tool(description="获取具体城市的天气",args_schema=WeatherInput)
def get_weather(city:str,unit:str,if_forcast:bool):
    result= f"2026-09-18{city}天气晴朗,9-16{unit},温度适宜，适合出门晒太阳"
    if if_forcast:
        result+="但是明天可能有雨，记得带伞"
    return result
#4 将函数绑定在模型上
@tool(description="查询当天的新闻")
def get_news():
    """
    查询当天的新闻
    """
    return "赛力斯华为分手，赛力斯股价堪忧"
@tool(description="用于发送邮件给自己的邮箱")
def send_to_my_email():
    """
    用于发送邮件给自己的邮箱
    """
    return "已成功发送邮件"

@tool(description="查询股价的信息")
def search_stock():
    """
    查询股价的信息
    """
    return "今天的A股涨势喜人啊~~~~~~~~"
@tool
def calculate(num1:int,num2:int):
    """
    执行数学计算的2位数的加法

    Args:
        num1:加数
        num2:被加数
    """

    return num1+num2

myAgent=create_agent(
    model=myModel,
    name="agent01",
    # system_prompt="你是1个专业的个人助手，查询消息的时候会使用已知的工具",
    checkpointer=InMemorySaver(),
    tools=[get_weather,get_news,calculate,search_stock],
    middleware=[
        tool_selector,

    ]
   )

response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你好哇,今天是2026-9-18，请问重庆明天的天气如何，今天的股价行情怎么样")
            ],

    },
       config=myConfig,
)
# last_msg=response["messages"][-1]
# rprint(last_msg)
for msg in response["messages"]:
    msg.pretty_print()
# print(last_msg.response_metadata.get("model_name"))





================================ Human Message =================================

你好哇,今天是2026-9-18，请问重庆明天的天气如何，今天的股价行情怎么样
================================== Ai Message ==================================
Name: agent01
Tool Calls:
  get_weather (call_85a8279d6a7f48ab9cf5e0ad)
 Call ID: call_85a8279d6a7f48ab9cf5e0ad
  Args:
    city: 重庆市
    unit: celsius
    if_forcast: True
  search_stock (call_1df6a9e7a4c24187a491f046)
 Call ID: call_1df6a9e7a4c24187a491f046
  Args:
================================= Tool Message =================================
Name: get_weather

2026-09-18重庆市天气晴朗,9-16celsius,温度适宜，适合出门晒太阳但是明天可能有雨，记得带伞
================================= Tool Message =================================
Name: search_stock

今天的A股涨势喜人啊~~~~~~~~
================================== Ai Message ==================================
Name: agent01

你好哇！👋 帮你查好啦～

## 🌤️ 重庆天气

**今天（2026-9-18）**：晴朗 ☀️
- 气温：9~16℃（摄氏度）
- 温度适宜，非常适合出门晒晒太阳～

**明天（2026-9-19）**：⚠️ **可能有雨**
- 出门记得带伞哦！☔

## 📈 今日股价行情

今天 A 股**涨势喜人**！

ContextEditingMiddleware中间件 上下文编辑中间件，提供了管理上下文的管理方式，通过更改发送给模型的消息列表来控制成本（实际上不会更改消息列表，因此只能通过token用量来推测是否对消息列表进行了裁剪）

In [65]:

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
tool_model = init_chat_model(
    model="qwen3.8-flash",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
#实例化的1个对象 工具选择类中间件
tool_selector=LLMToolSelectorMiddleware(
    model=tool_model,
    max_tools=1,
    always_include=["get_weather"]
)

myConfig={"configurable":{"thread_id":"1"}}

class WeatherInput(BaseModel):
    #如果在函数声明处指定了参数以及参数类型，并且启用了args_schema，那么则以args_schema指定的类型为准
    city: str=Field(
        description="具体的城市名称，例如'北京市'、'重庆市'、''天津市",
        default="北京市",
        min_length=2,
        max_length=10,
    )
    unit:Literal["celsius","fahrenheit","kelvin"]=Field(
        description="描述气温的单位，必须是摄氏度，凯文等单位"
    )
    if_forcast:bool=Field(
        default=False,
        description="是否预测未来1天的天气"
    )
    count:int=Field(description="调用的次数")
#3 声明1个函数（工具）
@tool(description="获取具体城市的天气",args_schema=WeatherInput)
def get_weather(city:str,unit:str,if_forcast:bool,count:int):
    #故意返回一段非常冗长、包含大量 Token 的文本，用于测试中间件的 Token 清理/截断功能
  return (f"当前是第 {count} 次调用工具, {city}今天天气晴朗"
        f"天气非常好，北风，非常适合出行，盼望着，盼望着，"
        f"春天来了。我喜欢春天，你喜欢吗，天气真的很不错"
        f"万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的"
        f"真不错，天气非常好，适合出行，这里token挺多的"
        f"可以出门玩，可以跑步，钓鱼，爬山，一切都很好哈哈哈")
#4 将函数绑定在模型上
@tool(description="查询当天的新闻")
def get_news():
    """
    查询当天的新闻
    """
    return "赛力斯华为分手，赛力斯股价堪忧"
@tool(description="用于发送邮件给自己的邮箱")
def send_to_my_email():
    """
    用于发送邮件给自己的邮箱
    """
    return "已成功发送邮件"

@tool(description="查询股价的信息")
def search_stock():
    """
    查询股价的信息
    """
    return "今天的A股涨势喜人啊~~~~~~~~"
@tool
def calculate(num1:int,num2:int):
    """
    执行数学计算的2位数的加法

    Args:
        num1:加数
        num2:被加数
    """

    return num1+num2

myAgent=create_agent(
    model=myModel,
    name="agent01",
    # system_prompt="你是1个专业的个人助手，查询消息的时候会使用已知的工具",
    checkpointer=InMemorySaver(),
    tools=[get_weather,get_news,calculate,search_stock],
    middleware=[
        tool_selector,
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=50,
                    keep=0
                )
            ]
        )

    ]
   )

#模拟3次对话
for i in range(3):
    print("--"*50,f"这是第{i}轮对话","--"*50)
    count=i+1
    response=myAgent.invoke(
        {
            # 严格校验，使用message少了1个s就会报错
            "messages":
                [
                    HumanMessage(f"第{i}次询问：你好哇,今天是2026-9-18，请问重庆明天的天气如何？一句话回答")
                ],

        },
           config=myConfig,
    )
    # last_msg=response["messages"][-1]
    # rprint(last_msg)
    for msg in response["messages"]:
        # msg.pretty_print()
        if isinstance(msg,AIMessage):
            if not msg.tool_calls:
                print(f"本次token用量：{msg.usage_metadata}")





---------------------------------------------------------------------------------------------------- 这是第0轮对话 ----------------------------------------------------------------------------------------------------
本次token用量：{'input_tokens': 898, 'output_tokens': 163, 'total_tokens': 1061, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 22}}
---------------------------------------------------------------------------------------------------- 这是第1轮对话 ----------------------------------------------------------------------------------------------------
本次token用量：{'input_tokens': 898, 'output_tokens': 163, 'total_tokens': 1061, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 22}}
本次token用量：{'input_tokens': 1140, 'output_tokens': 147, 'total_tokens': 1287, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 22}}
------------------------------------------------------------------------------------------------

作为上一轮次的对照组，以下代码取消了Context中间件

In [66]:

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
tool_model = init_chat_model(
    model="qwen3.8-flash",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
#实例化的1个对象 工具选择类中间件
tool_selector=LLMToolSelectorMiddleware(
    model=tool_model,
    max_tools=1,
    always_include=["get_weather"]
)

myConfig={"configurable":{"thread_id":"1"}}

class WeatherInput(BaseModel):
    #如果在函数声明处指定了参数以及参数类型，并且启用了args_schema，那么则以args_schema指定的类型为准
    city: str=Field(
        description="具体的城市名称，例如'北京市'、'重庆市'、''天津市",
        default="北京市",
        min_length=2,
        max_length=10,
    )
    unit:Literal["celsius","fahrenheit","kelvin"]=Field(
        description="描述气温的单位，必须是摄氏度，凯文等单位"
    )
    if_forcast:bool=Field(
        default=False,
        description="是否预测未来1天的天气"
    )
    count:int=Field(description="调用的次数")
#3 声明1个函数（工具）
@tool(description="获取具体城市的天气",args_schema=WeatherInput)
def get_weather(city:str,unit:str,if_forcast:bool,count:int):
    #故意返回一段非常冗长、包含大量 Token 的文本，用于测试中间件的 Token 清理/截断功能
  return (f"当前是第 {count} 次调用工具, {city}今天天气晴朗"
        f"天气非常好，北风，非常适合出行，盼望着，盼望着，"
        f"春天来了。我喜欢春天，你喜欢吗，天气真的很不错"
        f"万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的"
        f"真不错，天气非常好，适合出行，这里token挺多的"
        f"可以出门玩，可以跑步，钓鱼，爬山，一切都很好哈哈哈")
#4 将函数绑定在模型上
@tool(description="查询当天的新闻")
def get_news():
    """
    查询当天的新闻
    """
    return "赛力斯华为分手，赛力斯股价堪忧"
@tool(description="用于发送邮件给自己的邮箱")
def send_to_my_email():
    """
    用于发送邮件给自己的邮箱
    """
    return "已成功发送邮件"

@tool(description="查询股价的信息")
def search_stock():
    """
    查询股价的信息
    """
    return "今天的A股涨势喜人啊~~~~~~~~"
@tool
def calculate(num1:int,num2:int):
    """
    执行数学计算的2位数的加法

    Args:
        num1:加数
        num2:被加数
    """

    return num1+num2

myAgent=create_agent(
    model=myModel,
    name="agent01",
    # system_prompt="你是1个专业的个人助手，查询消息的时候会使用已知的工具",
    checkpointer=InMemorySaver(),
    tools=[get_weather,get_news,calculate,search_stock],
    middleware=[
        tool_selector,
    ]
   )

#模拟3次对话
for i in range(3):
    print("--"*50,f"这是第{i}轮对话","--"*50)
    count=i+1
    response=myAgent.invoke(
        {
            # 严格校验，使用message少了1个s就会报错
            "messages":
                [
                    HumanMessage(f"第{i}次询问：你好哇,今天是2026-9-18，请问重庆明天的天气如何？")
                ],

        },
           config=myConfig,
    )
    # last_msg=response["messages"][-1]
    # rprint(last_msg)
    for msg in response["messages"]:
        # msg.pretty_print()
        if isinstance(msg,AIMessage):
            if not msg.tool_calls:
                print(f"本次token用量：{msg.usage_metadata}")





---------------------------------------------------------------------------------------------------- 这是第0轮对话 ----------------------------------------------------------------------------------------------------
本次token用量：{'input_tokens': 839, 'output_tokens': 264, 'total_tokens': 1103, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 76}}
---------------------------------------------------------------------------------------------------- 这是第1轮对话 ----------------------------------------------------------------------------------------------------
本次token用量：{'input_tokens': 839, 'output_tokens': 264, 'total_tokens': 1103, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 76}}
本次token用量：{'input_tokens': 1231, 'output_tokens': 207, 'total_tokens': 1438, 'input_token_details': {'cache_read': 1024}, 'output_token_details': {'reasoning': 9}}
----------------------------------------------------------------------------------------------

FilesystemFileSearchMiddleware中间件 基于glob和grep检索工具，为Agent赋予本地文件搜索和分析能力，其中glob根据文件路径搜索，grep根据文件内容检索

In [67]:

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)

myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    middleware=[
        FilesystemFileSearchMiddleware(
            root_path="../todo_workspace", #搜索目录
            # 【可选】限制搜索的文件后缀，防止模型读取非代码或无关文件
            # allowed_extensions=[".py", ".ipynb", ".js", ".md"],  # 允许的文件类型
            # 是否启用 ripgrep 搜索引擎：
            # 设为 True 可以获得比原生 Grep 更快的性能（前提是系统已安装 ripgrep）
            # use_ripgrep=True,
            # 单个文件的最大读取限制 (单位MB)：防止读取超大型日志或二进制文件导致 OOM
            max_file_size_mb=10

        )
    ]
   )


response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你好哇,请你找到包含add函数的python文件")
            ],

    },

)
# last_msg=response["messages"][-1]
# rprint(last_msg)
for msg in response["messages"]:
    msg.pretty_print()






================================ Human Message =================================

你好哇,请你找到包含add函数的python文件
================================== Ai Message ==================================
Name: agent01

你好！我来帮你搜索包含 `add` 函数的 Python 文件。
Tool Calls:
  grep_search (call_0704351038704d708ecdb7e0)
 Call ID: call_0704351038704d708ecdb7e0
  Args:
    pattern: def\s+add\s*\(
    include: *.py
    output_mode: content
  grep_search (call_b026d5673d6b4c5fb2ba4d78)
 Call ID: call_b026d5673d6b4c5fb2ba4d78
  Args:
    pattern: def\s+add\w*\s*\(
    include: *.py
    output_mode: count
================================= Tool Message =================================
Name: grep_search

/celebrate1.py:1:def add(number1:int|float , number2:int|float):
================================= Tool Message =================================
Name: grep_search

/celebrate1.py:1
================================== Ai Message ==================================
Name: agent01

找到了！我来看看这个文件的具体内容：
Tool Calls:
  glob_searc